# 00 — Verificación acotada de conexión con ThetaData

**Objetivo.** Confirmar, con una ventana de cinco minutos de SPY, que el SDK remoto puede devolver barras, quotes, vencimientos, quotes de una call y una put del mismo strike, y griegas de primer orden si el permiso lo permite. Esto no descarga un historial ni construye una estrategia.

La clave se lee desde `.env` mediante la variable `THETADATA_API_KEY`; no se muestra ni se guarda. Las respuestas se archivan localmente bajo `data/.../raw/`, ignoradas por Git.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'scripts' / 'thetadata_connection_check.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'scripts' / 'thetadata_connection_check.py').exists(), 'No se localizó la raíz de opciones_notas.'
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.thetadata_connection_check import run_connection_check

result = run_connection_check(PROJECT_ROOT)
display(result['summary'])

,request,status,rows,failure_kind,exception_type
0,stock_history_ohlc,ok,6,None,None
1,stock_history_quote,ok,6,None,None
2,option_list_expirations,ok,2177,None,None
3,option_list_strikes,ok,163,None,None
4,option_history_quote,ok,12,None,None
5,option_history_greeks_first_order,ok,12,None,None


## Parámetros y selección

La fecha solicitada es el 24 de noviembre de 2026, de 10:00 a 10:05 en `America/New_York`. Para el subyacente se pidió explícitamente el feed `nqb`. El vencimiento debe quedar entre 27 y 33 días calendario y se elige el más cercano a 30; el strike es el más próximo al primer precio disponible al inicio de la ventana.

El timestamp se conserva tal como lo entrega ThetaData. En OHLC de 1m, representa la apertura y cubre operaciones desde ese instante hasta antes del siguiente minuto. En quotes de acciones, representa la última quote anterior al timestamp; en quotes de opciones, la última NBBO quote en el timestamp. Por tanto, no se equiparan sus semánticas.

In [2]:
display(result['selected'])
print('SDK:', result['metadata']['sdk_version'])
print('Python:', result['metadata']['python_version'])
print('Feed del subyacente:', result['metadata']['stock_feed'])

,spot_from_initial_stock_quote_mid,expiration,dte_calendar,strike,right_requested
0,719.685,2026-12-24,30,720.0,both


SDK: 1.0.18
Python: 3.13.2
Feed del subyacente: nqb


## Respuestas pequeñas del SDK

Las tablas siguientes son muestras de las respuestas devueltas. Si una petición falló, la tabla queda vacía y la primera tabla identifica si se trató de autenticación, permisos, disponibilidad o formato/transporte. Los Parquet `raw` conservan el DataFrame del SDK antes de añadir metadatos; el SDK remoto no expone aquí el cuerpo gRPC original.

In [3]:
for name in ('stock_ohlc', 'stock_quotes', 'expirations', 'strikes', 'option_quotes', 'greeks'):
    frame = result[name]
    print(f'\n{name}: {len(frame)} filas; columnas = {list(frame.columns)}')
    display(frame.head(8))


stock_ohlc: 6 filas; columnas = ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'count', 'vwap']


,timestamp,open,high,low,close,volume,count,vwap
0,2026-11-24 10:00:00-05:00,719.68,720.9300,719.50,719.57,251950,4384,720.11
1,2026-11-24 10:01:00-05:00,719.57,720.2600,719.31,719.51,120486,2580,720.01
2,2026-11-24 10:02:00-05:00,719.53,719.6889,718.90,719.29,103756,2001,719.86
3,2026-11-24 10:03:00-05:00,719.33,720.4600,719.33,720.16,163830,2902,719.83
4,2026-11-24 10:04:00-05:00,720.21,720.4600,719.81,720.35,100138,1971,719.87
5,2026-11-24 10:05:00-05:00,NaN,NaN,NaN,NaN,0,0,719.87



stock_quotes: 6 filas; columnas = ['timestamp', 'bid_size', 'bid_exchange', 'bid', 'bid_condition', 'ask_size', 'ask_exchange', 'ask', 'ask_condition']


,timestamp,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition
0,2026-11-24 10:00:00-05:00,40,1,719.68,0,40,7,719.69,0
1,2026-11-24 10:01:00-05:00,200,3,719.53,0,80,1,719.59,0
2,2026-11-24 10:02:00-05:00,40,1,719.45,0,40,1,719.48,0
3,2026-11-24 10:03:00-05:00,360,1,719.33,0,40,65,719.34,0
4,2026-11-24 10:04:00-05:00,40,7,720.13,0,40,1,720.19,0
5,2026-11-24 10:05:00-05:00,40,7,720.34,0,600,1,720.40,0



expirations: 2177 filas; columnas = ['symbol', 'expiration']


,symbol,expiration
0,SPY,2012-06-01
1,SPY,2012-06-08
2,SPY,2012-06-16
3,SPY,2012-06-22
4,SPY,2012-06-29
5,SPY,2012-07-06
6,SPY,2012-07-13
7,SPY,2012-07-21


... (2 outputs left out)


option_quotes: 12 filas; columnas = ['symbol', 'expiration', 'strike', 'right', 'timestamp', 'bid_size', 'bid_exchange', 'bid', 'bid_condition', 'ask_size', 'ask_exchange', 'ask', 'ask_condition']


,symbol,expiration,strike,right,timestamp,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition
0,SPY,2026-12-24,720.0,CALL,2026-11-24 10:00:00-05:00,45,22,18.93,50,60,11,19.16,50
1,SPY,2026-12-24,720.0,CALL,2026-11-24 10:01:00-05:00,45,22,18.83,50,297,6,19.06,50
2,SPY,2026-12-24,720.0,CALL,2026-11-24 10:02:00-05:00,45,22,18.82,50,261,6,19.05,50
3,SPY,2026-12-24,720.0,CALL,2026-11-24 10:03:00-05:00,2,7,18.71,50,1,65,18.92,50
4,SPY,2026-12-24,720.0,CALL,2026-11-24 10:04:00-05:00,88,42,19.09,50,287,6,19.33,50
5,SPY,2026-12-24,720.0,CALL,2026-11-24 10:05:00-05:00,60,73,19.17,50,111,65,19.41,50
6,SPY,2026-12-24,720.0,PUT,2026-11-24 10:00:00-05:00,131,1,18.47,50,1,73,18.67,50
7,SPY,2026-12-24,720.0,PUT,2026-11-24 10:01:00-05:00,62,22,18.55,50,45,22,18.77,50



greeks: 12 filas; columnas = ['symbol', 'expiration', 'strike', 'right', 'timestamp', 'bid', 'ask', 'delta', 'theta', 'vega', 'rho', 'epsilon', 'lambda', 'implied_vol', 'iv_error', 'underlying_timestamp', 'underlying_price']


,symbol,expiration,strike,right,timestamp,bid,ask,delta,theta,vega,rho,epsilon,lambda,implied_vol,iv_error,underlying_timestamp,underlying_price
0,SPY,2026-12-24,720.0,CALL,2026-11-24 10:00:00-05:00,18.93,19.16,0.5302,-0.3390,82.0754,29.7998,-31.3647,20.0427,0.2192,0.0,2026-11-24 10:00:00-05:00,719.68
1,SPY,2026-12-24,720.0,CALL,2026-11-24 10:01:00-05:00,18.83,19.06,0.5291,-0.3383,82.0779,29.7407,-31.2975,20.1039,0.2188,0.0,2026-11-24 10:01:00-05:00,719.56
2,SPY,2026-12-24,720.0,CALL,2026-11-24 10:02:00-05:00,18.82,19.05,0.5283,-0.3390,82.0797,29.6846,-31.2406,20.0782,0.2193,0.0,2026-11-24 10:02:00-05:00,719.46
3,SPY,2026-12-24,720.0,CALL,2026-11-24 10:03:00-05:00,18.71,18.92,0.5271,-0.3380,82.0812,29.6217,-31.1677,20.1605,0.2186,0.0,2026-11-24 10:03:00-05:00,719.33
4,SPY,2026-12-24,720.0,CALL,2026-11-24 10:04:00-05:00,19.09,19.33,0.5345,-0.3378,82.0587,30.0590,-31.6379,20.0380,0.2182,0.0,2026-11-24 10:04:00-05:00,720.16
5,SPY,2026-12-24,720.0,CALL,2026-11-24 10:05:00-05:00,19.17,19.41,0.5363,-0.3374,82.0482,30.1726,-31.7580,20.0307,0.2178,0.0,2026-11-24 10:05:00-05:00,720.37
6,SPY,2026-12-24,720.0,PUT,2026-11-24 10:00:00-05:00,18.47,18.67,-0.4700,-0.2870,82.0805,-29.3324,27.8062,-18.2183,0.2379,0.0,2026-11-24 10:00:00-05:00,719.68
7,SPY,2026-12-24,720.0,PUT,2026-11-24 10:01:00-05:00,18.55,18.77,-0.4710,-0.2875,82.0817,-29.3929,27.8592,-18.1648,0.2383,0.0,2026-11-24 10:01:00-05:00,719.56


## Límite de esta prueba

Esto verifica capacidad técnica y permisos para una consulta mínima. No verifica cobertura histórica general, calidad económica, hora de publicación EOD ni ejecutabilidad. Tampoco se interpretan bid/ask como una estrategia ni se inicia streaming.

### Notas personales

- 
- 